In [13]:
%pwd

'C:\\Users\\91971\\Desktop\\medical chatbot'

In [ ]:
import os
os.chdir('../')

In [14]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


c:\Users\91971\Desktop\medical chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
def pdf_loader(pdf_path):
    loader = DirectoryLoader(pdf_path, glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [16]:
data=pdf_loader('data')

In [18]:
len(data)

637

In [19]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document] :



    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document (
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [20]:
filtered_data = filter_to_minimal_docs(data)

In [22]:
len(filtered_data)

637

In [23]:
def text_splitter(docs: List[Document], chunk_size: int = 500, chunk_overlap: int = 20) -> List[Document]:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    return text_splitter.split_documents(docs)

In [29]:
text_chunks = text_splitter(filtered_data)

In [24]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():

    

    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(model_name=model_name)

    return embeddings

embedding = download_embeddings()

C:\Users\91971\AppData\Local\Temp\ipykernel_7456\946771443.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name)
c:\Users\91971\Desktop\medical chatbot\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\91971\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DIS

In [ ]:
embedding.embed_query("Hello world")

[-0.034477267414331436,
 0.031023245304822922,
 0.006735002622008324,
 0.02610897272825241,
 -0.03936199098825455,
 -0.16030247509479523,
 0.06692401319742203,
 -0.006441466510295868,
 -0.047450482845306396,
 0.014758829958736897,
 0.07087530195713043,
 0.05552758648991585,
 0.01919335499405861,
 -0.026251377537846565,
 -0.010109559632837772,
 -0.02694047801196575,
 0.022307416424155235,
 -0.02222663350403309,
 -0.1496925950050354,
 -0.01749303564429283,
 0.0076762172393500805,
 0.054352302104234695,
 0.003254439914599061,
 0.0317259281873703,
 -0.0846213549375534,
 -0.0294059868901968,
 0.051595572382211685,
 0.048124056309461594,
 -0.003314792178571224,
 -0.058279186487197876,
 0.04196929186582565,
 0.022210659459233284,
 0.128188818693161,
 -0.022338882088661194,
 -0.011656327173113823,
 0.06292837858200073,
 -0.03287631645798683,
 -0.0912260040640831,
 -0.031175334006547928,
 0.052699558436870575,
 0.04703483358025551,
 -0.08420311659574509,
 -0.0300561785697937,
 -0.02074487134814

In [26]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [27]:
from pinecone import Pinecone
pc=Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
pc

In [35]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot-1"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384, # Dimension of the embeddings
        metric= "cosine", # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1"))

index = pc.Index(index_name)

In [36]:
from langchain_pinecone import PineconeVectorStore

pinecone_vector_store = PineconeVectorStore(
    index=index,
    embedding=embedding
)

pinecone_vector_store.add_documents(text_chunks)

['3a7d325a-5c0f-42c6-bd07-b2bb0c83f86e',
 '9ac60d9d-9b1c-442a-b2d9-17de5c5201ce',
 'fba3e32d-be0e-49bd-8d09-e7962e879f97',
 'bfc0082a-ce57-4a3e-a399-741ba8568257',
 '84401b67-3711-4328-bf70-8d56d64a9d5b',
 '16125ec8-c0e9-4358-8a01-f3f6dec316aa',
 '6d6ebd08-7812-404d-ab0e-8c89ddcf3b27',
 'aef2d625-e054-4b9f-8a5e-54310bd7efb6',
 '5f3e68b8-323a-4f92-a6e5-d48ecaf93b08',
 '3c7391b9-e5ca-4f43-9a26-da046098cea9',
 '0a559219-de5b-4df1-ad92-936a4d92cd03',
 '142bc63e-21ff-4a46-bfbf-a8876abc9ea8',
 '04ffc718-8a65-473e-b5f0-13b4d5107501',
 '98e19f56-074b-43f9-9249-67a5a013df2d',
 '15fb3ecf-73d5-4fa1-b4fd-70f854b8a95d',
 '81ccabdb-91c2-44cb-a800-30ca00b595fe',
 'e10a23d4-e35a-4959-a712-d3a4ee162a76',
 'db8dc349-68ea-4724-9b3f-11042a6800ac',
 'f87f51fa-919c-4ad6-959b-1c915ea6713e',
 '40996b43-55f3-41fd-8a62-d1c0899f0d10',
 'f98dfa07-d671-4abf-8918-18d3225cfd8b',
 '1f4e0c4e-0dc3-475f-88ee-a694e6268127',
 'dc1e5797-cb01-42bc-86ba-76af2a42186d',
 'ecac42df-3d6d-432b-9b7c-6c13bbe1caa7',
 '0bbbf42c-65b4-

In [37]:
from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index
docsearch = PineconeVectorStore. from_existing_index(
index_name=index_name,
embedding=embedding
)



In [38]:
retriever=docsearch.as_retriever(search_kwargs={"k": 3},search_type="similarity",)

In [41]:
results = retriever.invoke("what is acne")

In [42]:
results

[Document(id='d6a0df72-91b2-453c-9ede-83ea79271f8f', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='5b73f05f-2618-4ca0-a0cd-303210a05791', metadata={'source': 'data\\Medical_book.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged with oil, dead skin\ncells, and bacteria.\nDescription\nAcne vulgaris, the medical term for common acne, is\nthe most common skin disease. It affects nearly 17 million\npeople in the United States. While acne can arise at any'),
 Document(id='47114cf2-d1a4-4d96-a92d-a91e5beb508e', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the gene

In [49]:
from langchain_groq import ChatGroq
load_dotenv()
os.environ.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [51]:
llm.invoke("what is acne").content

'**Acne** is a common skin condition that occurs when hair follicles become clogged with oil (sebum) and dead skin cells. It typically appears on the face, chest, back, and shoulders, and can range from mild pimples to severe cystic lesions.\n\n### Key Points\n\n| Aspect | Details |\n|--------|---------|\n| **Causes** | • Excess oil production<br>• Hair follicles clogged with dead skin cells<br>• Bacterial growth (especially *Cutibacterium acnes*)<br>• Hormonal changes (puberty, menstrual cycle, pregnancy, certain medications)<br>• Genetics |\n| **Common Types** | • **Comedonal** (blackheads and whiteheads)<br>• **Inflammatory** (papules, pustules, nodules, cysts) |\n| **Risk Factors** | • Age (most common in teens and young adults)<br>• Hormonal fluctuations<br>• Certain cosmetics or skincare products<br>• Stress, diet, and lifestyle |\n| **Symptoms** | • Red or inflamed bumps<br>• Pus-filled lesions<br>• Oily skin<br>• Possible scarring or hyperpigmentation after healing |\n| **Treat

In [53]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are a medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the user's question. If you don't know the answer, "
    "say that you don't know. Do not make up medical facts. "
    "Use a maximum of three sentences and keep the answer concise."
    "\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [54]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )



In [55]:
# 5. Create the RAG pipeline
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser( )
    )

In [57]:
rag_chain.invoke("what is acromegaly and gigantism")

'Acromegaly is a disorder caused by excess growth hormone released from the pituitary gland after the growth plates have closed, leading to enlarged bones and soft tissues in adults. Gigantism is the same hormone excess but occurring before the growth plates fuse, resulting in abnormally tall stature. Both conditions are rare, affecting about 50 people per million, and symptoms develop gradually, often delaying diagnosis.'